# Donation forecasting & allocation (CRISP-DM)

## 1. Problem framing
Development and finance teams need **forward-looking views** of giving and how gifts **spread across programs and safehouses**. **Stakeholders:** director of development, program finance, leadership reporting. **Predictive vs explanatory:** **Predictive** models answer “how much / how might allocation look for similar donors?” (generalization to new periods). **Explanatory** Ridge on **log1p(amount)** summarizes **associations** between donor history and amount—useful for narrative slides, not causal claims. **Justification:** Planning is inherently predictive; compliance and storytelling need transparent linear explanations.

## 2. Data acquisition, preparation & exploration
Tables: **`donations`**, **`supporters`**, **`donation_allocations`**, linked by `supporter_id` / `donation_id`. `data_prep.load_donation_tables` and `feature_engineering.build_supervised_rows` construct **time-ordered** rows so each target gift uses only **prior** history (no leakage). Exploration: distributions of amounts, recency/frequency features, allocation shares. **Reproducible pipeline:** `run_all.py`, `export_artifacts.py`, `config.py`.

## 3. Modeling & feature selection
**Amount:** compare algorithms in `train_predictive.train_compare_amount`; **allocation:** `allocation_model.train_allocation_forest`. **Explanatory:** `train_explanatory.run_explanatory`. Features chosen from RFM-style history and categorical supporter fields; selection via regularization and model defaults documented in code.

## 4. Evaluation & interpretation
**Metrics:** RMSE, MAE, R² for amount; allocation errors as appropriate. **Validation:** time-based split. **Business read:** errors mean budget variance—**false high** forecasts can over-commit programs; **false low** leaves capacity unused. Interpret coefficients as **conditional associations**.

## 5. Causal and relationship analysis
Coefficients and importances reflect **correlation structure** (wealth proxies, seasonality, unobserved donor intent). **Not** causal without experiments. **Theoretical plausibility:** past giving predicts future giving; still confounded by macro shocks. Predictive accuracy can coexist with **non-causal** explanations.

## 6. Deployment notes
**Artifacts:** `ml_pipeline_donation_forecasting/serialized_models/*.joblib` + metadata. **Export to .NET:** `refresh_ml_artifacts.py --donors-only` or `ml_backend_export.donors_backend` → `backend/Intex.API/App_Data/ml/donors/`. **UI:** supporter churn / donor ML surfaces in `SupportersListPage.tsx`, ML API `frontend/src/lib/mlApi.ts`, controllers under `backend/Intex.API/Controllers/Ml*.cs`.

**Leakage reminder:** Features use only donations **before** the target gift date; **time-ordered** train/test split.

## 1. Imports & path

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for d in [p, *p.parents]:
        if (d / "ml-pipelines").is_dir() and (d / "data" / "lighthouse_csv_v7").is_dir():
            return d
    raise FileNotFoundError(
        "Could not find repo root (need ml-pipelines/ and data/lighthouse_csv_v7/). "
        f"cwd={p}"
    )

REPO_ROOT = _find_repo_root(Path.cwd())
ML_PIPELINES_DIR = REPO_ROOT / "ml-pipelines"
if str(ML_PIPELINES_DIR) not in sys.path:
    sys.path.insert(0, str(ML_PIPELINES_DIR))

from ml_pipeline_donation_forecasting import config
from ml_pipeline_donation_forecasting.allocation_model import combine_amount_and_allocation, train_allocation_forest
from ml_pipeline_donation_forecasting.data_prep import load_donation_tables, parse_dates
from ml_pipeline_donation_forecasting.dataset_utils import feature_target_columns
from ml_pipeline_donation_forecasting.export_artifacts import export_all
from ml_pipeline_donation_forecasting.feature_engineering import build_supervised_rows
from ml_pipeline_donation_forecasting.train_explanatory import run_explanatory
from ml_pipeline_donation_forecasting.train_predictive import train_compare_amount

config.OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
config.SERIALIZED_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load data & build supervised rows

In [ ]:
tabs = parse_dates(load_donation_tables(config.DEFAULT_DATA_DIR))
df, meta = build_supervised_rows(tabs["donations"], tabs["donation_allocations"], tabs["supporters"])
print("rows", len(df), "supporters", meta["n_supporters"])
print(meta["program_areas"])
df[["y_amount", "target_donation_date"]].head()

## 3. EDA — amount distribution & skew

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df["y_amount"].hist(bins=40, ax=ax, color="steelblue")
ax.set_title("Next donation amount (PHP) — skewed")
ax.set_xlabel("PHP")
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / "fig_amount_hist.png", dpi=120)
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
np.log1p(df["y_amount"]).hist(bins=30, ax=ax, color="darkorange")
ax.set_title("log1p(next amount)")
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / "fig_log_amount_hist.png", dpi=120)
plt.show()

## 4. Correlation (numeric features only)

In [ ]:
num, cat, _y = feature_target_columns(df, meta)
cm = df[num].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap="vlag", center=0)
plt.title("Numeric feature correlations")
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / "fig_corr.png", dpi=120)
plt.show()

## 5. Allocation vs amount (bivariate)

In [ ]:
pa = meta["allocation_targets_program"][0]
if pa in df.columns:
    plt.figure(figsize=(5, 4))
    plt.scatter(df[pa], df["y_amount"], alpha=0.35)
    plt.xlabel(pa)
    plt.ylabel("Next amount PHP")
    plt.title("Share vs amount (associational)")
    plt.tight_layout()
    plt.savefig(config.OUTPUTS_DIR / "fig_alloc_vs_amount.png", dpi=120)
    plt.show()

## 6. Explanatory OLS (log amount)

In [ ]:
_, coef_df = run_explanatory(df, meta)
display(coef_df.head(20))

## 7. Predictive amount models

In [ ]:
amt = train_compare_amount(df, meta)
display(amt["all_results"])
print("Best:", amt["best_name"])

## 8. Allocation model + combined funding (test rows)

In [ ]:
alloc = train_allocation_forest(df, meta, amt["numeric_features"], amt["categorical_features"])
import numpy as np
from ml_pipeline_donation_forecasting.evaluate import regression_metrics
pred_log = np.clip(amt["best_pipeline"].predict(amt["X_test"]), 0, 25)
y_amt_pred = np.expm1(pred_log)
print("Amount holdout:", regression_metrics(amt["y_test"], y_amt_pred))
print("Allocation program MAE:", alloc["program_area_metrics"])
comb = combine_amount_and_allocation(y_amt_pred, alloc["Y_pred"], meta)
print(json.dumps(comb, indent=2))

## 9. Export joblib + JSON

In [ ]:
info = export_all(df, meta)
print(info)